# 04 - Body segmentation (M5)
**Function in the system:** the mask produces the background-free image for the listing, and the
mask area over the frame area tells staff when the animal is too small or cut off.
We compare a zero-shot SAM baseline (prompted with the body box) with a fine-tuned YOLO11s-seg.
Metrics are reported with and without the trimap border band, which is ambiguous by design.
Revision of 19 October 2026: Drive is mounted before repository discovery; the common-subset comparison is saved as CSV/JSON on future runs. Training, predictions and recorded model outputs below come from the existing 18–19 October T4 run. Common-subset means were independently recomputed from its saved per-image CSVs; the modified bootstrap and saving statements are not a new GPU reproduction.


In [ ]:
# Setup — mount Drive before resolving the repository.
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

def _resolve_spv_repo(search_root: Path, explicit: str | None = None) -> Path:
    def valid(path):
        return (path / "src" / "spv").is_dir() and (path / "configs" / "config.yaml").is_file()
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not valid(candidate):
            raise FileNotFoundError(f"SPV_REPO is not a project root: {candidate}")
        return candidate
    candidates = set()
    for pattern in ("smart-pet-vision", "*smart-pet-vision", "*/smart-pet-vision", "*/*/smart-pet-vision"):
        candidates.update(path.resolve() for path in search_root.glob(pattern) if valid(path))
    if valid(search_root):
        candidates.add(search_root.resolve())
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one smart-pet-vision repository under {search_root}; found {len(candidates)}. "
            "Add the shared folder shortcut to My Drive, or set SPV_REPO to its exact path."
        )
    return candidates.pop()

if IN_COLAB:
    REPO = _resolve_spv_repo(Path("/content/drive/MyDrive"), os.environ.get("SPV_REPO"))
else:
    explicit = os.environ.get("SPV_REPO")
    local_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                       if (path / "src" / "spv").is_dir()), Path.cwd())
    REPO = _resolve_spv_repo(local_root, explicit)
os.chdir(REPO)
if IN_COLAB:
    !pip -q install -r requirements.txt
sys.path.insert(0, str(REPO / "src"))
from spv.common import load_config, seed_everything, save_json, load_json
cfg = load_config()
if IN_COLAB:
    cfg["paths"]["raw"] = Path("/content/data/raw")
    cfg["paths"]["processed"] = Path("/content/data/processed")
seed_everything(cfg["seed"])
RUNS = Path(cfg["paths"]["runs"])
import torch
print("Repository:", REPO)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


In [ ]:
from spv import data
df = data.ensure_data(cfg)          # first run: download + clean + split (about 10 min)
IMG = data.dataset_dir(cfg) / "images"
PROC = Path(cfg["paths"]["processed"])
print(len(df), "images after cleaning")

In [ ]:
from spv import segment
OUT = RUNS / "segmentation"; OUT.mkdir(parents=True, exist_ok=True)
test = df[df.split == "test"]
sam_subset = test.sample(n=min(300, len(test)), random_state=cfg["seed"])
sam = segment.sam_masks(sam_subset, IMG)              # downloads SAM 2.1-B weights
sam_summary = segment.run_test("sam_baseline", sam, sam_subset, IMG, PROC, OUT); sam_summary

In [ ]:
SEG_YAML = PROC / "yolo_segment" / "data.yaml"
seg_weights = segment.train(cfg, SEG_YAML)
save_json({"weights": os.path.relpath(seg_weights)}, OUT / "weights.json")
masks = segment.yolo_masks(seg_weights, [IMG / f"{n}.jpg" for n in test.image], conf=0.25,
                           imgsz=cfg["segmentation"]["imgsz"])
yolo_summary = segment.run_test("yolo_seg", masks, test, IMG, PROC, OUT)
coco = segment.yolo_coco_metrics(seg_weights, SEG_YAML, cfg["segmentation"]["imgsz"])
save_json(coco, OUT / "yolo_seg_coco.json"); yolo_summary, coco

In [ ]:
import pandas as pd
same = {k: v for k, v in masks.items() if k in set(sam_subset.image)}
yolo_on_subset, _ = segment.evaluate(same, sam_subset, PROC)
metric_names = ["mIoU", "mean_dice", "mIoU_no_border", "mean_dice_no_border"]
comparison = pd.DataFrame({"SAM (box prompt)": sam_summary,
                           "YOLO11s-seg (same 300)": yolo_on_subset}).loc[metric_names]
comparison.to_csv(OUT / "same_subset_comparison.csv", index_label="metric")
save_json({"split": "test", "n": int(len(sam_subset)), "sampling_seed": cfg["seed"],
           "image_ids": sam_subset.image.tolist(),
           "sam": {key: sam_summary[key] for key in metric_names},
           "yolo": {key: yolo_on_subset[key] for key in metric_names}},
          OUT / "same_subset_comparison.json")
comparison.round(4)


In [ ]:
from IPython.display import Image as Show, display
display(Show(str(OUT / "figures" / "yolo_seg_failures.jpg"), width=900))

## Ground-truth mask check and the role of SAM (added 24 Oct 2026)
Oxford-IIIT Pet already provides pixel trimaps, so the ground-truth masks are the trimap foreground plus the
ambiguous border. They are checked visually below (green = foreground, yellow = border band). SAM 2.1-B, prompted
with the ground-truth body box, serves as an independent check of these masks and as the zero-shot baseline; it is
not a deployable component because it needs the ground-truth box. The team-set masks were drafted automatically and
reviewed and corrected in CVAT by the four team members, one part of about 100 photos each (dataset card).

In [ ]:
import pandas as pd
from IPython.display import Image as Show, display
from spv import segment
OUT = RUNS / "segmentation"
segment.gt_overlay_sheet(df[df.split == "test"], IMG, PROC, OUT / "figures" / "gt_mask_check.jpg")
display(Show(str(OUT / "figures" / "gt_mask_check.jpg"), width=900))
sam_pi = pd.read_csv(OUT / "sam_baseline_per_image.csv")
agree = {"images": int(len(sam_pi)),
         "sam_iou_ge_0.8": int((sam_pi.iou >= 0.8).sum()),
         "sam_iou_no_border_ge_0.9": int((sam_pi.iou_no_border >= 0.9).sum()),
         "sam_iou_lt_0.5": int((sam_pi.iou < 0.5).sum())}
save_json(agree, OUT / "sam_trimap_agreement.json")
agree

**Reading the results.** On the same 300 test images the fine-tuned YOLO11s-seg is ahead of the SAM baseline with the border included (mIoU 0.925 against 0.812, Dice 0.960 against 0.883) and on border-ignored Dice (0.964 against 0.956), while SAM has the higher border-ignored mIoU (0.948 against 0.934). SAM receives the ground-truth box, so it is an oracle-prompted reference rather than a deployable component. Its worst cases are close-ups where the box covers nearly the whole frame and SAM segments the background; the fine-tuned model's worst cases spill onto blankets or clothing of similar colour (gallery above: red = predicted only, green = missed, yellow = agreement).